# KG chat session -> turns + knowledge graph

The knowledge-graph-populating sibling of **[`chat_session.ipynb`](chat_session.ipynb)**:
`KgChatSession` (from **[`chat_sessions.py`](chat_sessions.py)**) is a `ChatSession` that, for
every new **human** turn:

1. Runs `LLM_EventExtraction.annotate_new_turn()` (from
   `events_from_chat/llm_event_triples_openai_pydantic.py`) to extract semantic-role
   annotations (activities/conditions, roles, perspective) for just that one turn.
2. If that produced anything, pushes it into the knowledge graph right away with
   `populate_ekg_from_annotations()` (from `events_from_chat/populate_ekg.py`).
3. **If new triples were pushed**, looks for knowledge-graph gaps around the activity/activities
   that turn just added, using `kg_gap_finder.py` restricted to those subject URIs (e.g. "most
   other 'exercise' activities have a `location`, this one doesn't").
4. **If a gap was found**, turns it into a natural-language follow-up question with
   `LLMTripleReplier` (from `chat_from_kg/llm_triple_replier.py`) and uses that as the agent's
   reply -- a small, KG-grounded "curiosity" question instead of a generic LLM reply.
5. **Otherwise** (no new triples, or no gaps for them) falls back to the default `agent_fn` reply,
   same as plain `ChatSession`.

The agent's own reply is then recorded and annotated/pushed too (turn 2, 4, 6, ... in the
conversation), same as the human's.

`annotate_new_turn()` is the incremental sibling of `annotate_all_turns_in_conversation()`: instead
of resetting its history and re-annotating every turn from scratch on every call, it keeps the
extractor's `_history`/`_known_activities` alive across calls, so each new turn costs exactly one
OpenAI call while the model still sees every real prior turn as context (so cross-turn activity
coreference — reusing the same `activity_id` for "it"/"that" — still works correctly). One
`LLM_EventExtraction` instance holds the state for exactly one conversation; call
`extractor.reset_conversation()` before reusing an instance for a different one.

**Before running this:** `llm_event_triples_openai_pydantic.py` reads `OPENAI_API_KEY` at *import*
time (module-level), and `populate_ekg_from_annotations()`/`kg_gap_finder.py` need a reachable
knowledge-graph SPARQL endpoint (e.g. a local GraphDB `event_sandbox` repository) — set
`KG_ADDRESS` below to point at yours. Both only get imported the first time you construct a
`KgChatSession` (see `chat_sessions._load_kg_dependencies()`), not just from running the import
cell below.

**On `gap_threshold`:** a gap only counts if at least that fraction of an activity's peers (same
type, e.g. "exercise") share the predicate it's missing. In a small/sparse graph, few predicates
reach a strict majority, so `KgChatSession` defaults to `0.3` rather than `kg_gap_finder`'s own
default of `0.6` — tune it per how populated your graph is.


In [1]:
import time

from chat_sessions import KgChatSession, save_turns, DEFAULT_GAP_ACTIVITY_TYPES

KG_ADDRESS = "http://localhost:7200/repositories/event_sandbox"
KG_LOG_DIR = "kg_logs"
GAP_THRESHOLD = 0.3

KG_ADDRESS = "http://localhost:7200/repositories/diabetes_event_details_and_types"
GAP_THRESHOLD = 0.6

# A FRESH id every run, not a fixed constant -- see the markdown above the next cell for why
# reusing one (e.g. a hardcoded 999999) silently corrupts the graph across separate
# conversations. Pin it to a fixed int instead only if you deliberately want to CONTINUE a
# previous run's own conversation (same chat id => same activity_id numbering => new turns
# extend, rather than collide with, that same run's own activities).
CHAT_ID = int(time.time())

# Which activity TYPES gap-finding is allowed to ask follow-up questions about -- an activity of
# any other type (or one whose type wasn't captured at all) still gets pushed to the graph and
# still shows up in the graph panel, it just never triggers a gap-driven question. Local names as
# they appear in the graph's own rdf:type triples (see chat_sessions.DEFAULT_GAP_ACTIVITY_TYPES
# for the full explanation, including why this is a config-time list here rather than a live
# slider like Gap sensitivity -- and how to check what other types actually exist in *your* own
# graph via a SPARQL "SELECT ?t (COUNT(*) AS ?n) WHERE {?s a ?t} GROUP BY ?t"). Pass
# gap_activity_types=None below instead to consider every activity type, unrestricted.
GAP_ACTIVITY_TYPES = DEFAULT_GAP_ACTIVITY_TYPES


## Run a live, KG-populating chat

Opens the conversation in its own window (`kg_chat_gui.py`'s `ChatWindow`), instead of the
`input()`-in-the-cell loop `ChatSession.run_interactive()` uses: the transcript (both **you**
and the **agent**) scrolls in that one window, and you type your next line into the entry box
built into the bottom of that *same* window -- no separate popup, no notebook `input()` prompt.

Every turn is still annotated and pushed to `KG_ADDRESS` as it happens, same as before. Each
agent turn is labeled **[KG]** (green) when it came from a knowledge-graph gap question, or
**[LLM]** (orange) when it's the default LLM reply -- read straight from
`kg_session.reply_sources`. Requires `OPENAI_API_KEY` and a reachable knowledge graph.

**Why `CHAT_ID` is generated fresh, not a fixed number:** each activity a turn mentions gets a KG
subject URI of the form `.../n2mu/chat<CHAT_ID>.<N>`, where `N` is just a per-conversation
counter that a brand-new `KgChatSession`/extractor always restarts at 1 -- it has no way to know
what a *previous, unrelated* run already put in the (shared, persistent) knowledge graph under
that same chat id. Reusing a fixed `CHAT_ID` (e.g. a hardcoded `999999`) across separate runs
therefore doesn't start a fresh graph each time -- it makes every run's first activity land on
the exact same URI as every other run's first activity (same for the second, third, ...), so
their triples all silently pile up on one shared node. Concretely: if an earlier run's first turn
was "I feel so tired lately" and today's is "I had dinner", both become
`chat999999.1` in the graph, which then shows `rdfs:label` "tired" *and* "dinner" on the same
node, peer-comparison gap-finding sees a bloated and irrelevant peer group, and the graph panel
renders a confusing mix of both conversations -- there's nothing wrong with the session or the
panel at that point, the KG data itself is just corrupted by the id collision. Picking `CHAT_ID`
fresh (this cell uses the current Unix timestamp) makes every run's activities land on their own,
never-before-used URIs. Deliberately reuse a fixed id only if you actually want to *continue* one
specific earlier conversation and are sure nothing else has ever run with that same id.

The **Gap sensitivity** slider adjusts `gap_threshold` live, mid-conversation -- drag it down to
make the agent curious about more things, up to make it ask only about what most peers agree on.
It overrides `GAP_THRESHOLD` from the moment you move it (and drops any gaps already queued at
the old setting, so the change bites on the very next lookup). A **Text size** slider (starting
at 16pt) live-resizes the whole conversation panel at once.

Since `KG_ADDRESS` points at a GraphDB repository, the window also splits: the chat stays on the
left, and a **graph panel** appears on the right -- an actual node-link diagram drawn right in
the window: the activity the conversation is currently about as one node in the middle, every
triple pushed for it as a labeled line out to its own object node, colored by its RDF namespace
-- **red for `n2mu`**, **blue for `gaf`**, **green for `grasp`** (this project's own ontology's
three namespaces), a distinct color per any other namespace, gray for a plain literal (see the
small legend in the panel). Its own **Font size** slider (also starting at 16pt, independent of
the conversation's) resizes the diagram's labels and node sizes. A dropdown switches between
every activity mentioned so far. Tkinter can't embed GraphDB's own interactive D3 view (it's a
JS app, and Tkinter has no browser engine), so this is a simpler diagram drawn from the same
data -- **Open in GraphDB ↗** is still there for the real, fully interactive "Visual graph" (drag
nodes, expand further, ...) in your browser, and **auto-open new activities** opens that
automatically the first time each new activity comes up, if you'd rather not click it yourself.

**While it's running**, each turn prints a diagnostic block to this cell's own output: what it
pushed to the knowledge graph, any fresh gap query it ran (with a B/D/E breakdown), and which gap
its reply was actually about -- see `kg_session.turn_log` below, or `chat_sessions.LOG_TURNS` to
turn the printing off.

**To stop:** click **Quit**, or type "quit"/"bye"/... The window closes and the cell finishes,
and the conversation, a statistics summary, and this per-turn log are written to three
timestamped JSON files under `notebooks/chat_logs/` (`chat<chat>_turns_<stamp>.json` /
`chat<chat>_stats_<stamp>.json` / `chat<chat>_gaplog_<stamp>.json`) -- their paths are printed
below the cell. `run_gui()` then returns `kg_session.turns`, so everything below still works
unchanged; pass `save_dir=None` to skip the automatic save.


In [ ]:
from kg_chat_gui import run_gui

kg_session = KgChatSession(
    chat=CHAT_ID,
    human="Mehmet",
    kg_address=KG_ADDRESS,
    log_dir=KG_LOG_DIR,
    gap_threshold=GAP_THRESHOLD,
    gap_activity_types=GAP_ACTIVITY_TYPES,
)
kg_turns = run_gui(kg_session)


Inspect what was extracted, pushed, and where each agent reply came from:


In [ ]:
print(f"{len(kg_session.turns)} turns, {len(kg_session.annotations)} annotated, "
      f"{len(kg_session.kg_pushes)} pushes to the knowledge graph")
print("reply sources:", kg_session.reply_sources)
kg_session.kg_pushes


`kg_session.turn_log` has the same per-turn breakdown that was printed live above, for each turn:
what it pushed, any fresh gap query it ran, and which gap its reply was about (`None` for a
default, non-gap-driven reply):


In [ ]:
kg_session.turn_log


In [4]:
save_turns(kg_session.turns, "kg_turns.json")

Wrote 26 turns to kg_turns.json
